In [ ]:
# ============================================================
# Dashboard interactivo: Poblacion por edad - Las Americas
# Adaptado para Google Colab
# ============================================================

# Habilitar el administrador de widgets en Colab
from google.colab import output
output.enable_custom_widget_manager()

import math
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
from IPython.display import display
import plotly.graph_objects as go

# ------------------------------------------------------------
# 1) Datos (modelo demografico)
# ------------------------------------------------------------
YEARS = list(range(1950, 2101))
AGES = list(range(0, 101))

def birth_rate(year):
    """Tasa relativa de nacimientos por año (logistica decreciente)."""
    y0, k, high, low = 1985.0, 0.045, 6.0, 1.55
    return low + (high - low) / (1 + math.exp(k * (year - y0)))

def survival(age):
    """Curva de supervivencia tipo Gompertz-Makeham simplificada."""
    return math.exp(-((age / 88.0) ** 4.3))

def build_dataset():
    data = {}
    for y in YEARS:
        raw = []
        for a in AGES:
            by = min(max(y - a, 1900), 2100)
            raw.append(birth_rate(by) * survival(a))
        total = sum(raw)
        data[y] = [100 * v / total for v in raw]
    return data

DATA = build_dataset()

def hex_to_rgba(hex_color, alpha):
    hex_color = hex_color.lstrip("#")
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

BLUE = "#4a7fa5"
GREEN = "#3fb63f"

# ------------------------------------------------------------
# 2) Bloques de texto
# ------------------------------------------------------------
top_bar = widgets.HTML(
    "<div style='border-top:4px solid #f0b429; margin-bottom:10px;'></div>"
)
title_html = widgets.HTML(
    f"<h2 style='color:{BLUE}; font-weight:500; margin:4px 0 0 0;'>Poblacion por edad</h2>"
)
subtitle_html = widgets.HTML(
    f"<div style='color:{BLUE}; text-align:center; font-size:15px; margin-bottom:6px;'>Las Americas</div>"
)

def badge_html(value, color):
    return (
        f"<div style='min-width:58px; text-align:center; background:{color}; "
        f"color:#fff; border-radius:6px; padding:4px 10px; font-weight:600; "
        f"font-size:14px; display:inline-block;'>{value}</div>"
    )

# ------------------------------------------------------------
# 3) Widgets de control
# ------------------------------------------------------------
year_slider = widgets.IntSlider(
    value=2026, min=min(YEARS), max=max(YEARS), step=1,
    description="Anio", continuous_update=True,
    layout=Layout(width="380px"), style={"description_width": "70px"},
)
year_badge = widgets.HTML(badge_html(year_slider.value, BLUE))

age_slider = widgets.IntSlider(
    value=25, min=0, max=100, step=1,
    description="Edad seleccionada", continuous_update=True,
    layout=Layout(width="380px"), style={"description_width": "140px"},
)
age_badge = widgets.HTML(badge_html(age_slider.value, GREEN))

area_color_picker = widgets.ColorPicker(
    concise=False, description="Color area:", value="#78C878",
    style={"description_width": "90px"},
)
line_color_picker = widgets.ColorPicker(
    concise=False, description="Color linea:", value="#3FB63F",
    style={"description_width": "90px"},
)
highlight_color_picker = widgets.ColorPicker(
    concise=False, description="Color resaltado:", value="#2E8B2E",
    style={"description_width": "90px"},
)

readout = widgets.HTML()

# ------------------------------------------------------------
# 4) Figura como FigureWidget
# ------------------------------------------------------------
init_year, init_age = year_slider.value, age_slider.value
init_pcts = DATA[init_year]
init_pct_sel = init_pcts[init_age]

fig = go.FigureWidget(
    data=[
        go.Scatter(
            x=init_pcts, y=AGES, mode="lines", fill="tozerox",
            fillcolor=hex_to_rgba(area_color_picker.value, 0.35),
            line=dict(color=hex_to_rgba(line_color_picker.value, 0.8), width=1),
            hovertemplate="Edad %{y}: %{x:.2f}%<extra></extra>",
            name="Poblacion",
        ),
        go.Scatter(
            x=[0, init_pct_sel], y=[init_age, init_age], mode="lines",
            line=dict(color=highlight_color_picker.value, width=5),
            hoverinfo="skip", showlegend=False,
        ),
    ],
    layout=go.Layout(
        xaxis=dict(title="", ticksuffix="%", range=[0, 3.3], gridcolor="#e9edf0",
                   zeroline=False, fixedrange=True),
        yaxis=dict(title="Edad", range=[0, 100], gridcolor="#e9edf0",
                   zeroline=False, fixedrange=True),
        dragmode=False,
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(t=20, r=30, l=50, b=50), showlegend=False,
        width=850, height=560,
        annotations=[dict(
            x=0, y=min(init_age + 2.5, 97),
            text=f"{init_age}-year-olds The Americas: {init_pct_sel:.1f}%",
            showarrow=False, xanchor="left", font=dict(size=13, color="#444"),
        )],
    ),
)
fig._config = {"displayModeBar": False, "scrollZoom": False}

# ------------------------------------------------------------
# 5) Callback de actualizacion
# ------------------------------------------------------------
def update(*_):
    year = year_slider.value
    age = age_slider.value
    pcts = DATA[year]
    pct_sel = pcts[age]

    year_badge.value = badge_html(year, BLUE)
    age_badge.value = badge_html(age, GREEN)
    readout.value = (
        f"<div style='font-size:14px; color:#555; margin-top:6px;'>"
        f"Personas de <b>{age}</b> anios en <b>{year}</b>: "
        f"<b>{pct_sel:.2f}</b> % de la poblacion total de Las Americas.</div>"
    )

    with fig.batch_update():
        fig.data[0].x = pcts
        fig.data[0].fillcolor = hex_to_rgba(area_color_picker.value, 0.35)
        fig.data[0].line.color = hex_to_rgba(line_color_picker.value, 0.8)

        fig.data[1].x = [0, pct_sel]
        fig.data[1].y = [age, age]
        fig.data[1].line.color = highlight_color_picker.value

        fig.layout.annotations = [dict(
            x=0, y=min(age + 2.5, 97),
            text=f"{age}-year-olds The Americas: {pct_sel:.1f}%",
            showarrow=False, xanchor="left", font=dict(size=13, color="#444"),
        )]

for w in (year_slider, age_slider, area_color_picker, line_color_picker, highlight_color_picker):
    w.observe(update, names="value")

update()

# ------------------------------------------------------------
# 6) Layout final
# ------------------------------------------------------------
controls_box = VBox(
    [
        HBox([year_slider, year_badge]),
        HBox([age_slider, age_badge]),
        HBox([area_color_picker, line_color_picker, highlight_color_picker]),
    ],
    layout=Layout(background_color="#f7f9fa", padding="14px 16px", margin="10px 0"),
)

footer = widgets.HTML(
    "<div style='font-size:11px; color:#999; text-align:center; margin-top:14px;'>"
    "Modelo demografico ilustrativo (transicion demografica aproximada), no corresponde a "
    "cifras oficiales exactas de la ONU. Rango: 1950-2100.</div>"
)

display(VBox([top_bar, title_html, subtitle_html, controls_box, readout, fig, footer]))